# 리포트 5 — 우리 커널 — 무엇이고, 무엇이 아닌가

> SBR + 물리광학이 무엇을 계산하고 무엇을 **계산하지 않는지**를 정의하고, 해석해가 있는 과녁(구·평판·이면각)으로 잰다.

이 권은 아래 절로 이루어진다. 각 절은 **한 일 · 결과 · 방법 · 재현** 을 자기 앞에 달고 있어, 필요한 절만 따로 읽어도 된다.

| 절 | 무엇을 말하나 | 만든 곳 |
|---|---|---|
| 1 | 가림 판정은 Sionna 광선엔진이 하고, 면적분은 우리 커널이 한다 | `_parts/18_kernel-what.ipynb` |
| 2 | 스톡 솔버와 맞대면 «면이 많아서 에코가 커진다» 가설은 반증되고, 런타임의 96.9% 는 호스트가 쓴다 | `_parts/19_kernel-vs-stock.ipynb` |
| 3 | 수신 방향 그림자 광선을 켜면 상반성 위반이 9.69 → 8.24 dB 로 내려간다 | `_parts/20_bistatic-exit.ipynb` |
| 4 ⭐ | 해석 PO 구 대비 구현오차는 kr 전 구간에서 λ/16 격자 0.201 dB · 생산 λ/12 격자 0.254 dB 안이다 | `_parts/21_kernel-vs-reference.ipynb` |
| 5 | PO 유효 무릎을 부품 폭으로 옮기면 어느 부품이 어느 밴드에서 떨어지는지가 보인다 | `_parts/22_po-knee.ipynb` |
| 6 | 커널이 아직 못 하는 것은 편파 분리·PTD·재테셀레이션·다중반사 Γ(θ) 넷이고, 각각의 크기를 적었다 | `_parts/23_kernel-open-items.ipynb` |

⭐ 표시한 절 하나만 읽어도 이 권의 결론은 선다.

숫자는 전부 계산 결과 JSON(원장)에서 주입된다 — 절 끝 «출처» 표가 그 파일과 키다. 원장이 다시 계산되면 빌더를 돌리는 것만으로 본문 숫자가 따라 바뀐다.

전체 목차는 [reports/README.md](README.md) 이고, 열일곱 권의 지도는 [리포트 1 «이 연구가 묻는 것과 답한 방식»](01_map.ipynb) 다.


---

## 절 1. 가림 판정은 Sionna 광선엔진이 하고, 면적분은 우리 커널이 한다



> ### 한 일
> **상용 고주파 솔버의 순서 그대로 광선으로 조명면을 찾고 그 면 위에서 부품별 재질 PO 를 적분해 σ 를 냈다.**

### 결과
1. 첫 충돌 탐색과 가림은 Sionna 가 이미 들고 있는 Mitsuba/OptiX 엔진이 하고, 표면전류 적분과 σ 출력은 우리가 얹는다 — 그 문서에 `physical optics` 는 0 회 [^1] 나온다.
2. 조명원을 방위 280° [^2] · 고각 15° [^3] 에 두면 조명원을 향한 외피의 29 [^4]~47% [^5] 가 기체 자신에 가려 있다.
3. «가림 [dB]» 은 **두 팔의 차**다 — P(순수 PO · 점구름 λ/7 · 가림 없음)의 방위평균 σ 에서 B(SBR+PO · 광선격자 λ/12)를 **불투명**으로 돌린 값을 뺀 것이 최대 6.63 dB [^6] (Matrice 4E ⭐ [^7], 닫힌 동체)이고, 열린 프레임인 S1000+ [^8] 에서는 0.11 dB [^9] 다. 기체 7 종 [^10] 전부에서 이 차가 이산화 바닥(P 팔 한쪽을 λ/7↔λ/12 로 돌린 폭, 최대 0.071 dB [^11]) 위에 있다 — 원장이 «가림 효과» 라고 부르기 위해 건 필요조건을 일곱 기체가 전부 통과한다 [^12].
4. 금속 4그룹만 남긴 메쉬의 방위평균 σ 가 전체의 112% [^13] 다 — 코히런트 합이라 100 % 를 넘는다.
5. 그 광선 격자를 자세마다 다시 정의하면 로터 사이에 가짜 결합이 생긴다 — 가산성 잔차가 격자를 얼렸을 때 8.3e-16 [^14] (기계정밀도)이고 움직이는 격자에서 1.78 [^15] 다. 얼리면 **대역밖**(블레이드 끝 도플러 1229 Hz [^16] 보다 높은 주파수 전부) 절대 전력이 λ/12 에서 13.1 dB [^17] 내려간다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| ① 조명면 찾기 | Sionna 의 Mitsuba/OptiX 광선엔진을 그대로 부른다 — 첫 충돌 탐색과 자기가림 판정이 그쪽 몫이다 |
| ② 면적분 | 그 면 위에서 부품별 재질 PO 를 적분한다 (`src/rcs_sbr.py` `rcs_sbr()`) — E = Σ \|Γᵢ(θᵢ)\| e^{j2k pᵢ·û} d², σ = 4π\|E\|²/λ² |
| 셸 투과 | 얇은 유전체 셸 뒤의 금속(배터리·PCB)을 코히런트 합산한다 (동 `penetrate=True`) |
| 가림의 크기 | 같은 자세를 **다른 팔**로 다시 적분한다 — P(순수 PO 점구름 λ/7, 가림 없음)와 B(SBR+PO 광선격자 λ/12)를 불투명으로 돌린 값의 방위평균 σ 차이를 기체마다 잰다. 셸까지 통과시킨 생산 B 와 P 의 차는 표의 «P − B(생산)» 열이다. 이산화 바닥은 P 팔 한쪽을 λ/7↔λ/12 로 돌린 폭이고 그 옆에 나란히 싣는다 |
| 격자를 무엇에 매나 | 격자 중심·반경·칸수를 자세마다 다시 잡는 팔과 한 판으로 얼린 팔을 같은 씬·같은 자세열에 나란히 태운다 |

### 재현

```bash
PYTHONPATH=src python src/make_report02_target.py --derive-only
PYTHONPATH=src python src/build_part04_kernel.py
```

| | |
|---|---|
| 출력 | `outputs/report02_derived.json`, `outputs/prior_settled_sionna.json`, `outputs/report3_rt.json`, `outputs/sbr_grid_convergence.json`, `outputs/outofband_power.json`, `outputs/verify_frozen_grid.json`, `outputs/md_classify_verify.json` |
| 소요 | 약 2분 (GPU 0장 — 원장 조립이다) |
| 비고 | σ 격자 자체의 재생성은 `benchmark/rcs_anchor.py` 가 맡는다 |

---


## 두 낱말을 먼저 푼다

**PO** 는 물리광학(physical optics)이다 — 빛이 닿는 면에 흐르는 전류를 근사식으로 바로 적어 넣고 그 면을 훑어 더해 산란을 내는 방법이다. **SBR** 은 광선을 쏴서 튀기며 그 면이 어디인지 찾는 방법(shooting-and-bouncing rays)이다.

상용 고주파 RCS 솔버(FEKO/CST SBR+)의 순서 그대로다 — **① 광선으로 실제 조명면을 찾고 ② 그 위에서 PO 표면적분**(`src/rcs_sbr.py` `rcs_sbr()`). 레이다식이 표적 산란과 전파 경로를 두 양으로 쓰는 그대로, **σ 는 이 커널이 내고 경로와 환경은 그 엔진이 낸다**.


## 누가 무엇을 하나

| 단계 | 무엇을 | 누가 |
|---|---|---|
| 첫 충돌 탐색 · 가림 | 어느 면이 실제로 조명되는가 | 🟢 Sionna 의 Mitsuba/OptiX 광선엔진 |
| 재질 \|Γ(θ)\| | 수직입사 보정값 × 각도 모양(TE·TM 전력평균, `ANGLE_GAMMA=1` 기본). 1회 반사 경로 넷이 같은 함수를 쓴다 — 다중반사 경로가 서 있는 자리는 **절 6** 이다 | 🟢 Sionna 재질표(`src/materials.py` `MATERIALS`) + 🔵 각도 모양 (`src/rcs_sbr.py` `ANGLE_GAMMA`) |
| PO 면적분 → σ | E = Σ \|Γᵢ(θᵢ)\| e^{j2k pᵢ·û} d², σ = 4π\|E\|²/λ² | 🔵 우리 (`src/rcs_sbr.py` `rcs_sbr()`) |
| 셸 투과 | 얇은 유전체 셸 뒤 금속(배터리·PCB)의 코히런트 합 | 🔵 우리 (동 `penetrate=True`) |


## 왜 우리가 얹어야 하나

Sionna 는 광선을 쏘고 튀긴다 — 기술보고서(v1.2, 59쪽)에 SBR 이 48 회 [^18] 나오고 우리도 그 엔진을 그대로 부른다. 같은 문서에서 `physical optics` 0 회 [^1] · `radar cross section` 0 회 [^19] · `surface current` 0 회 [^20] 이고, 거친 면은 정규화 산란패턴을 쓰는 경험 모델이다 — 그 셈이 어디서 끝나는지는 [리포트 2 «스톡 엔진이 하는 일과 안 하는 일»](02_stock-engine.ipynb) 가 인자 목록까지 해부했다.

ITU `metal` 의 산란계수 S = 0.0 [^21] 이라 스톡 산란 모델이 금속에서 내놓는 항은 0 이고, 우리 σ 는 면적분에서 창발한다. 금속 4그룹(모터·배터리·PCB·카메라)만 남긴 메쉬의 방위평균 σ 는 전체의 112% [^13] 다.


## PO 적분이 실제로 올라타는 면은 어디까지인가

![mesh_compare_material_shadow](../outputs/figures/mesh_compare_material_shadow.png)

**그림 1.** PO 적분이 실제로 올라타는 면은 어디까지인가?

조명원을 방위 280° [^2] · 고각 15° [^3] 에 두었다 — 방위 72 점 [^22] 스윕에서 7기체 평균 그늘비율의 중앙값에 가장 가까운 방위이고, 규칙이 고른다. 가림 판정은 생산 SBR 이 쓰는 그림자광선 그대로다(`rcs_sbr._exit_visible()`). 그림의 색은 재질이 아니라 조명 상태다.


**표.** 열 이름의 두 팔 — P = 순수 PO(점구름 λ/7), B = SBR+PO(광선격자 λ/12) 이고, B 는 불투명(셸 투과 없음)과 생산(셸 투과) 두 설정으로 선다.

> 원장 정의 그대로다 — 가림 [dB] = 가림 없는 PO 방위평균 σ − 불투명 SBR σ. 셸 투과 [dB] 는 거기서 유전체 셸을 통과시켜 내부 금속을 코히런트 합산했을 때의 차이. 합 = 생산 커널과 순수 PO 의 차이. 이산화 바닥은 PO 를 λ/7↔λ/12 로 돌렸을 때의 폭이다. [^23]

| 기체 | 외피 그늘 | 가림 [dB] | 셸 투과 [dB] | P − B(생산) [dB] | 이산화 바닥 [dB] | 생산 σ [dBsm] |
|---|---|---|---|---|---|---|
| Mini 5 Pro ⭐ | 41 % | +5.98 | +3.66 | +2.32 | 0.014 | -22.0 |
| Mavic 4 Pro | 29 % | +3.75 | +2.55 | +1.20 | 0.021 | -18.2 |
| Matrice 4E ⭐ | 35 % | +6.63 | +3.72 | +2.90 | 0.021 | -18.9 |
| Phantom 4 | 36 % | +5.90 | +4.06 | +1.84 | 0.056 | -19.9 |
| X500 V2 | 47 % | +1.07 | +0.00 | +1.07 | 0.037 | -16.8 |
| Typhoon H (H480) | 39 % | +2.91 | +1.51 | +1.40 | 0.014 | -15.8 |
| S1000+ | 42 % | +0.11 | -0.19 | +0.30 | 0.071 | -12.3 |

출처 [^24]


## 두 팔의 차가 얼마인가

P(순수 PO · 점구름 λ/7 · 가림 없음)와 B(SBR+PO · 광선격자 λ/12)를 불투명으로 돌린 값의 방위평균 σ 차이가 6.63 dB [^6] (Matrice 4E ⭐ [^7], 닫힌 동체)까지 가고, 열린 프레임인 S1000+ [^8] 에서는 0.11 dB [^9] 다. 기체 7 종 [^10] 전부에서 이 차가 이산화 바닥(최대 0.071 dB [^11]) 위에 있다 — 그 바닥은 **P 팔 한쪽**을 λ/7↔λ/12 로 돌린 폭이고, 원장은 이것을 «가림 효과» 라고 부르기 위한 필요조건으로 쓴다 [^12]. B 팔 광선격자의 자기 불확도는 같은 통계(방위 72 점 [^22] 평균)로 재는 것이 다음 단계 표에 있다.

⚠ 이 표와 그림은 2026-08-04 [^25] 형상 정정 **전** 메쉬 기준이고, 2026-08-07 10:58:22 [^26] Γ(θ) 각도 모양(기본 켬) **이전** 커널의 산출이다 — 가림 최대치를 내는 Matrice 4E 와 X500 V2 가 그 정정을 받은 기체이고, 닫힌 동체의 가림은 셸 형상에 직접 걸린다. 생산 σ 열도 두 축 같은 이유로 재계산 대상이다.


## 격자를 자세마다 다시 정의하면 무엇이 생기나

광선 격자는 표적 앞에 세우는 평면 자다 — 중심 ctr, 반경 Rout, 한 변의 칸수 n 셋이 그것을 정한다. 생산 경로는 그 셋을 **자세마다 bbox 에서 다시 잡는다**. 관절이 도는 로터에서는 bbox 가 자세마다 숨쉬므로 자도 같이 흔들린다.

| 흔들리는 것 | 무엇이 흔들리나 (λ/12 · matrice4e · 4096 자세) | 무엇이 실리나 |
|---|---|---|
| 위상 원점 | ctr 이 시선방향으로 39.9 mm [^27] p-p 돌아다닌다 = 5.85 rad [^28] p-p | 진폭은 4e-16 [^29] 안에서 불변인 채 **위상만** 흔들린다 — 정지한 동체를 시선방향으로 숨쉬게 만드는 것과 같다 |
| 표본 격자 | n = ceil(2Rout/d) 가 정수라 100 [^30]~131 [^31] 사이를 오가고 4095 스텝 중 1636 번 [^32] (40 %) 튄다 | 서브셀 오프셋 표준편차 0.2912 [^33] 는 균등분포 1/√12 = 0.2887 과 넷째 자리까지 같다 — 자세마다 굴리는 **백색 주사위**다 |
| 히트 집합 | 조명된 광선이 평균 610.5 개 [^34], 자세간 상대 표준편차 0.0378 [^35] | 자세별 히트 수 계열 [^36] 의 최대−최소가 102 개(평균의 17 %) 다 — 어느 면이 세어지는가가 자세마다 갈린다 |


## 결정적 검사 — 가산성

서로 가리지 않는 로터의 PO 면적분은 E(φ₁..φ₄) = E₀ + Σ ΔE_j(φ_j) 로 정확히 쪼개진다. 로터 넷을 따로 돌린 합과 넷을 함께 돌린 장의 차이를 잔차로 쓴다 — 이 잣대에는 창도 평활도 분모도 안 들어간다.

| 격자 | 가산성 잔차 (중앙값) | 읽는 법 |
|---|---|---|
| 얼린 판 한 장 | 8.3e-16 [^14] ~ 1.2e-15 [^37] | 기계정밀도 — 정리가 그대로 성립한다 |
| 자세마다 다시 정의 | 0.089 [^38] ~ 1.78 [^15] | O(1) — 물리적으로 결합할 수 없는 로터 사이에 결합이 생긴다 |

그 가짜 결합이 변조로 실린다 — 기체별로 +3.7 [^39] ~ +23.3 dB [^40] 다. 교차 증거로, 광선을 안 쓰는 독립 엔진(순수 PO)과의 대역 안 스펙트럼 일치가 0.440 [^41] 에서 0.953 [^42] 으로 오른다.


## 얼리면 무엇이 오고 무엇을 잃나

잣대를 먼저 정의한다 — **대역밖 전력** P_out 은 슬로타임 스펙트럼에서 블레이드 끝 도플러 f_tip = 1229 Hz [^16] 보다 높은 주파수의 |X(f)|² 를 그대로 더한 값이다 [^43] — 평활도 비율도 안 들어가고, 정규화가 필요하면 분모를 이름에 박아 전체 전력으로 나눈다 [^44].

판 하나를 잡아 4096 자세에 그대로 쓰면 그 대역밖 절대 전력이 λ/12 에서 13.1 [^17] · λ/32 에서 20.1 dB [^45] 내려간다. ⭐ 격자 사다리의 세 팔(생산 · 위상고정 · 얼림 [^46])을 예측 기울기 ≈ −2 [^47] 에 맞대면, div ≥ 12 에서 잰 기울기가 생산 -0.56 [^48] (R² 0.946 [^49]) · 위상고정 -2.33 [^50] (R² 0.848 [^51]) · 얼림 -2.19 [^52] (R² 0.998 [^53]) 다 — 예측 위에 서는 것은 위상고정과 얼림 둘이고, 얼린 팔이 적합도와 절대 바닥(λ/12 에서 P_out 1.3e-08 [^54] 대 위상고정 1.2e-07 [^55])에서 앞선다. 생산 격자는 λ/12 → λ/32 로 촘촘히 해도 2.3 dB [^56] 만 내려간다 — 바닥의 지배 원인이 광선 밀도가 아니라는 뜻이다.

| 대가 | 크기 | 무엇을 뜻하나 |
|---|---|---|
| 광선 수 | 얼린 판이 자세 평균 대비 1.108 배 [^57] | 전 자세를 덮는 판이라 평균보다 크다 — 비용 +10.8 % |
| 디더 평균 | 얼린 장과 생산 장의 레벨 차가 3.35 dB [^58] p-p | 자세별 무작위 오프셋은 사실상 몬테카를로 평균이다. 얼리면 오프셋 한 판에 절대 레벨이 걸린다 — 절대 σ 는 정적 경로에서 가져오고 얼린 복소장은 **모양**에만 쓴다 |
| ⭐그 편향이 삼키는 것 | 판을 반 칸 옮기면 절대 레벨이 3.45 dB [^59] p-p, **두 팔의 차**(가림 축)가 4.16 dB [^60] p-p | 두 팔이 같은 판을 써도 기하가 달라 편향이 공통모드로 빠지지 않는다 — 차가 원본보다 더 흔들린다. 가림 dB 의 **크기**는 [리포트 8 절 5 «동체가 날개를 가리면 변조 깊이와 레벨이 함께…»](08_3_pattern.ipynb) 에서 판 앙상블 평균이 설 때까지 보류다 |
| 대역 안도 같이 내려간다 | 블레이드 대역 전력 중앙값 -13.9 dB [^61] | 대역밖만 내려가는 것이 아니다 — 아래 절이 그 몫이 잡음이었는지를 광선을 안 쓰는 엔진으로 판정한다 |
| 판을 미리 잡는 일 | 덮개 여유 최소 120.5 mm [^62] | 자세열을 먼저 훑어야 판이 나온다 — 스트리밍으로는 못 잡는다 |

커널의 기본값은 `grid_ref=None` 이다 — 그 값이면 배선 전 커널과 36 [^63]/36 [^64] 비트 동일이라 (최대 상대오차 0.0 [^65]) 옛 원장이 그대로 선다. 판을 잡는 쪽은 호출자다 — 슬로타임 경로(`src/microdoppler.py`)는 로터 한 바퀴의 합집합 경계상자로 판 한 장을 만들어 넘기고, 스위치 `SIONNA2_FREEZE_GRID` 가 그 켬·끔을 정한다(기본 켬). 판이 자세를 못 덮으면 커널이 예외를 던진다 [^66].

⭐ 리포트 8 계열의 마이크로도플러 원장은 얼린 판으로 **다시 났다**(27 열 · 옛 열은 `outputs/prefreeze/` 에 사본으로 남아 있다). 전후 비교와 그 판의 구체적인 수치는 [리포트 8-2 «광선 격자를 어디에 매나»](08_2_engines.ipynb) 가 낸다. ⚠ 아직 옛 판인 것은 바이스태틱 스윕 원장(`report07b_bistatic_md`)과 PO 대조 원장(`report15_po_control`)이다. 여기 숫자는 격자 사다리와 회귀 게이트 자신의 것이다 — 리포트 8 계열의 재계산 결과를 인용하지 않는다.


## 얼리기가 «신호» 도 깎았나 — 광선을 안 쓰는 엔진에게 묻는다

대역밖 전력이 내려간 것은 좋은 소식이다. 그런데 같은 재계산에서 **블레이드 대역 안**의 전력도 함께 내려갔다 — 중앙값 -13.9 dB [^61] 다. 그 몫이 표적의 진짜 운동이었다면 얼리기는 물리를 지운 것이다. 그래서 판정을 **광선 격자를 안 쓰는 엔진**에게 맡긴다 — 순수 PO 는 점구름 면적분이라 격자가 없고, 이번 재계산에서 비트 그대로였다 [^67].

| 잣대 | 얼리기 전 | 얼린 뒤 | 무엇을 뜻하나 |
|---|---|---|---|
| 블레이드 대역이 순수 PO 보다 몇 dB 위인가 | +20.3 dB [^68] | +4.0 dB [^69] | PO **밑으로** 내려간 열이 0 [^70] 개다 — 깎인 것은 잉여였다 |
| 같은 대역 복소 파형이 PO 와 닮은 정도 | 0.34 [^71] | 0.75 [^72] | 12 [^73]/13 [^74] 열에서 올랐다 — 전력이 줄면서 닮음이 오르면 줄어든 것은 잡음이다 |
| 플래시 대조비 (봉우리 ÷ 바닥) | — | +3.0 dB [^75] (중앙값) | 18 [^76]/27 [^77] 열에서 올랐다 — 자의 흔들림이 골짜기를 메우고 있었다는 뜻이다 |
| 플래시 주파수 추정의 상대오차 | 0.31% [^78] | 0.31% [^79] | 포락 자기상관으로 읽은 f_flash — 두 판 다 예측 위에 선다 |
| 스펙트럼 가장자리 ÷ f_tip (운동학이 맞으면 1) | 4.33 [^80] | 2.96 [^81] | 순수 PO 는 0.93 [^82] 다 — ⚠14 [^83] 열은 얼린 뒤 1 에서 더 멀어진다. 폭 지표는 PO 열에서 읽는다 |

⚠ **대가는 비율 쪽에 있다.** 절대 대역밖 전력은 내려가지만 블레이드 대역이 더 많이 내려가는 열이 있어서, «신호 대 비물리 잔차»(P_out/P_in)로 읽으면 동체가 든 15 [^84] 열 중 13 [^85] 열이 -8.8 dB [^86] → -6.2 dB [^87] 로 **나빠진다**(순수 PO 는 -29.6 dB [^88]). 잉여가 사라진 만큼 남은 잔차가 상대적으로 커 보이는 것이고, 그 잔차는 여전히 물리가 아니다 — 지금 판이 마이크로도플러에 줄 수 있는 동적범위의 상한이 여기 있다.

반대로 블레이드만 남긴 12 [^89] 열은 빗살 대조비가 12 [^90]/12 [^89] 열에서 +7.8 dB [^91] 올라온다 — 신호가 약할수록 얼리기가 크게 남는다 [^92].

⚠ 심판이 아직 붙지 않은 자리가 하나 남는다 [^93] — 2 초 호버 두 열은 순수 PO 대조 팔을 기다린다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 얼린 판으로 **바이스태틱** 스윕 원장과 PO 대조 원장을 다시 낸다 | 모노 쪽은 갈아탔고(리포트 8-2), 남은 두 원장만 아직 옛 판이라 두 편의 절대값을 이어 붙여 읽을 수 없다 | `src/microdoppler.py`(판을 넘기는 쪽) → [리포트 8 절 2 «시간표본마다 자세를 새로 놓고 다시 쏘아 슬로…»](08_3_pattern.ipynb) |
| B 팔 광선격자의 자기 불확도를 P 팔 바닥과 **같은 통계**(방위 72점 평균 σ)로 잰다 | 가림 축의 판정 바닥이 두 팔 모두에서 서고, 어느 기체가 판정 안에 드는지가 확정된다 | `src/viz_mesh_material.py` 가림 축 → outputs/report02_derived.json `occlusion.floor_db` |
| 정정된 메쉬로 가림 표와 생산 σ 를 같은 설정에서 다시 낸다 | 형상 정정이 가림과 σ 를 어느 방향으로 얼마나 옮기는지가 기체별로 확정된다 | [^94] |
| 같은 메쉬를 스톡 경로 솔버에 그대로 넣고 무엇이 나오는지 잰다 | 우리 커널이 스톡 위에 얹은 항이 무엇인지가 나란히 확정된다 | **절 2** «스톡 솔버와 맞대면 «면이 많아서 에코가 커진…» |
| 수신 방향 그림자 광선을 켜고 바이스태틱으로 넓힌다 | 출사 쪽 가림이 상반성 위반을 얼마나 줄이는지가 확정된다 | **절 3** «수신 방향 그림자 광선을 켜면 상반성 위반이…» |
| PO 면적분을 디바이스 커널로 옮긴다 | 전격자 재생성 비용이 확정된다 — 지금은 호스트가 대부분을 쓴다 | **절 2** «스톡 솔버와 맞대면 «면이 많아서 에코가 커진…» |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 94개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/prior_settled_sionna.json` | `word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.physical optics` | 0 |
| [^2] | `outputs/report02_derived.json` | `occlusion.az_deg` | 280 |
| [^3] | `outputs/report02_derived.json` | `occlusion.el_deg` | 15 |
| [^4] | `outputs/report02_derived.json` | `occlusion.shadow_min_pct` | 29.06 |
| [^5] | `outputs/report02_derived.json` | `occlusion.shadow_max_pct` | 46.54 |
| [^6] | `outputs/report02_derived.json` | `occlusion.max_db` | 6.626 |
| [^7] | `outputs/report02_derived.json` | `occlusion.max_drone` | Matrice 4E ⭐ |
| [^8] | `outputs/report02_derived.json` | `occlusion.min_drone` | S1000+ |
| [^9] | `outputs/report02_derived.json` | `occlusion.min_db` | 0.1111 |
| [^10] | `outputs/report02_derived.json` | `occlusion.n_above_floor` | 7 |
| [^11] | `outputs/report02_derived.json` | `occlusion.floor_max_db` | 0.07061 |
| [^12] | `outputs/mesh_compare_material.json` | `airframes.mini5pro.sigma.caveat` | PO 와 SBR 은 이산화가 다르다(점구름 λ/7 vs 광선격자 λ/12·jitter 2). dis… |
| [^13] | `outputs/report3_rt.json` | `C_metal.metal_share_pct` | 112 |
| [^14] | `outputs/md_classify_verify.json` | `grid_pinning.matrice4e.pinned.additivity_residual_median` | 8.282e-16 |
| [^15] | `outputs/md_classify_verify.json` | `grid_pinning.matrice4e.moving.additivity_residual_median` | 1.781 |
| [^16] | `outputs/outofband_power.json` | `_meta.f_tip_hz` | 1229 |
| [^17] | `outputs/outofband_power.json` | `freeze_verdict.gains_db.12` | 13.09 |
| [^18] | `outputs/prior_settled_sionna.json` | `word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.SBR or shooting-and-bouncing` | 48 |
| [^19] | `outputs/prior_settled_sionna.json` | `word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.radar cross section` | 0 |
| [^20] | `outputs/prior_settled_sionna.json` | `word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.surface current` | 0 |
| [^21] | `outputs/report3_rt.json` | `C_metal.itu_metal_S` | 0 |
| [^22] | `outputs/report02_derived.json` | `occlusion.n_az_sweep` | 72 |
| [^23] | `outputs/report02_derived.json` | `occlusion.definition` | 가림 [dB] = 가림 없는 PO 방위평균 σ − 불투명 SBR σ. 셸 투과 [dB] 는 거기서… |
| [^24] | `outputs/report02_derived.json` | `occlusion.rows` | (7행 표) |
| [^25] | `outputs/meshfix_applied.json` | `_meta.date` | 2026-08-04 |
| [^26] | `outputs/angle_gamma_impact.json` | `_meta.generated` | 2026-08-07 10:58:22 |
| [^27] | `outputs/sbr_grid_convergence.json` | `grid_wander.ctr_u_ptp_mm` | 39.89 |
| [^28] | `outputs/sbr_grid_freeze_review.json` | `R4_phase_only_arm.ctr_dot_u_ptp_rad` | 5.852 |
| [^29] | `outputs/sbr_grid_freeze_review.json` | `R4_phase_only_arm.rows[1].max_abs_change` | 3.906e-16 |
| [^30] | `outputs/sbr_grid_convergence.json` | `grid_wander.per_div.12.n_min` | 100 |
| [^31] | `outputs/sbr_grid_convergence.json` | `grid_wander.per_div.12.n_max` | 131 |
| [^32] | `outputs/sbr_grid_convergence.json` | `grid_wander.per_div.12.n_changes` | 1636 |
| [^33] | `outputs/sbr_grid_convergence.json` | `grid_wander.per_div.12.subcell_off_e1_std_frac` | 0.2912 |
| [^34] | `outputs/adv_grid_freeze_audit.json` | `audit_2_signal_loss.rows[1].n_lit_prod_mean` | 610.5 |
| [^35] | `outputs/adv_grid_freeze_audit.json` | `audit_2_signal_loss.rows[1].n_lit_prod_relstd` | 0.03777 |
| [^36] | `outputs/sbr_grid_convergence.npz` | `n_lit_div12` | (4096행 표) |
| [^37] | `outputs/md_classify_verify.json` | `grid_pinning.mini5pro.pinned.additivity_residual_median` | 1.167e-15 |
| [^38] | `outputs/md_classify_verify.json` | `grid_pinning.s1000plus.moving.additivity_residual_median` | 0.08905 |
| [^39] | `outputs/md_classify_verify.json` | `grid_pinning.matrice4e.spurious_modulation_db` | 3.651 |
| [^40] | `outputs/md_classify_verify.json` | `grid_pinning.phantom4.spurious_modulation_db` | 23.31 |
| [^41] | `outputs/sbr_grid_convergence.json` | `in_band_fidelity.rows[1].cos_prod_vs_po` | 0.4402 |
| [^42] | `outputs/sbr_grid_convergence.json` | `in_band_fidelity.rows[1].cos_froz_vs_po` | 0.9535 |
| [^43] | `outputs/outofband_power.json` | `new_definition.headline` | P_out = Σ \|X(f)\|² over \|f\| > f_tip   (평활 없음, 원 주기도) |
| [^44] | `outputs/outofband_power.json` | `new_definition.normalization` | 필요할 때만, **전체 전력**으로: frac_of_total = P_out / P_tot. 분모를… |
| [^45] | `outputs/outofband_power.json` | `freeze_verdict.gains_db.32` | 20.13 |
| [^46] | `outputs/sbr_grid_convergence.json` | `_meta.arms` | (3항목 묶음) |
| [^47] | `outputs/sbr_grid_convergence.json` | `_meta.prediction` | 실루엣이 격자를 가로지르며 광선이 토글 → 잡음 ∝ d² → 대역밖 비율의 log-log 기울기 ≈… |
| [^48] | `outputs/outofband_power.json` | `convergence.prod.slope_ge12` | -0.5604 |
| [^49] | `outputs/outofband_power.json` | `convergence.prod.r2_ge12` | 0.9463 |
| [^50] | `outputs/outofband_power.json` | `convergence.phase.slope_ge12` | -2.33 |
| [^51] | `outputs/outofband_power.json` | `convergence.phase.r2_ge12` | 0.848 |
| [^52] | `outputs/outofband_power.json` | `convergence.froz.slope_ge12` | -2.191 |
| [^53] | `outputs/outofband_power.json` | `convergence.froz.r2_ge12` | 0.9981 |
| [^54] | `outputs/outofband_power.json` | `convergence.froz.P_out_per_div.12` | 1.259e-08 |
| [^55] | `outputs/outofband_power.json` | `convergence.phase.P_out_per_div.12` | 1.152e-07 |
| [^56] | `outputs/outofband_power.json` | `convergence.prod.drop_db_div12_to_div32` | 2.297 |
| [^57] | `outputs/verify_frozen_grid.json` | `gate2_frozen_grid_invariant.extra_ray_cost` | 1.108 |
| [^58] | `outputs/verify_frozen_grid.json` | `field_level.froz_vs_prod_level_db_ptp` | 3.351 |
| [^59] | `outputs/freeze_plate_sensitivity.json` | `verdict.abs_level_plate_ptp_db` | 3.452 |
| [^60] | `outputs/freeze_plate_sensitivity.json` | `verdict.occlusion_level_plate_ptp_db` | 4.163 |
| [^61] | `outputs/freeze_signal_loss.json` | `summary.P_in_delta_db_median` | -13.87 |
| [^62] | `outputs/verify_frozen_grid.json` | `gate3_coverage.margin_min_mm` | 120.5 |
| [^63] | `outputs/verify_frozen_grid.json` | `gate1_bit_identity.n_bit_identical` | 36 |
| [^64] | `outputs/verify_frozen_grid.json` | `gate1_bit_identity.n_cases` | 36 |
| [^65] | `outputs/verify_frozen_grid.json` | `gate1_bit_identity.max_rel_err` | 0 |
| [^66] | `outputs/verify_frozen_grid.json` | `negative_controls.too_small_msg` | sbr_field: 얼린 격자가 이 자세를 못 덮는다 — 여유 e1 -161.21 · e2 +43.… |
| [^67] | `outputs/freeze_signal_loss.json` | `_meta.independent_judge_ko` | 순수 PO 열(`C_po_locked`·`D_po_spread`·`report07_three_eng… |
| [^68] | `outputs/freeze_signal_loss.json` | `summary.P_in_excess_over_po_db.before_median` | 20.3 |
| [^69] | `outputs/freeze_signal_loss.json` | `summary.P_in_excess_over_po_db.after_median` | 4.019 |
| [^70] | `outputs/freeze_signal_loss.json` | `summary.P_in_excess_over_po_db.n_below_po_after` | 0 |
| [^71] | `outputs/freeze_signal_loss.json` | `summary.blade_coh_vs_po.before_median` | 0.3429 |
| [^72] | `outputs/freeze_signal_loss.json` | `summary.blade_coh_vs_po.after_median` | 0.7489 |
| [^73] | `outputs/freeze_signal_loss.json` | `summary.blade_coh_vs_po.n_improved` | 12 |
| [^74] | `outputs/freeze_signal_loss.json` | `summary.blade_coh_vs_po.n_judged` | 13 |
| [^75] | `outputs/freeze_signal_loss.json` | `summary.flash_contrast_db.delta_median` | 2.979 |
| [^76] | `outputs/freeze_signal_loss.json` | `summary.flash_contrast_db.n_improved` | 18 |
| [^77] | `outputs/freeze_signal_loss.json` | `summary.flash_contrast_db.n_series` | 27 |
| [^78] | `outputs/freeze_signal_loss.json` | `summary.f_flash_relerr_abs_median.before` | 0.00313 |
| [^79] | `outputs/freeze_signal_loss.json` | `summary.f_flash_relerr_abs_median.after` | 0.00313 |
| [^80] | `outputs/freeze_signal_loss.json` | `summary.width_ratio_20db.before_median` | 4.329 |
| [^81] | `outputs/freeze_signal_loss.json` | `summary.width_ratio_20db.after_median` | 2.965 |
| [^82] | `outputs/freeze_signal_loss.json` | `summary.width_ratio_20db.po_median` | 0.93 |
| [^83] | `outputs/freeze_signal_loss.json` | `summary.width_ratio_20db.n_farther_from_one` | 14 |
| [^84] | `outputs/freeze_signal_loss.json` | `summary.by_group.with_body.n` | 15 |
| [^85] | `outputs/freeze_signal_loss.json` | `summary.by_group.with_body.n_oob_worse` | 13 |
| [^86] | `outputs/freeze_signal_loss.json` | `summary.by_group.with_body.oob_over_in_db_before_median` | -8.814 |
| [^87] | `outputs/freeze_signal_loss.json` | `summary.by_group.with_body.oob_over_in_db_after_median` | -6.245 |
| [^88] | `outputs/freeze_signal_loss.json` | `summary.oob_over_in_db.po_median` | -29.63 |
| [^89] | `outputs/freeze_signal_loss.json` | `summary.by_group.blade_only.n` | 12 |
| [^90] | `outputs/freeze_signal_loss.json` | `summary.by_group.blade_only.n_comb_improved` | 12 |
| [^91] | `outputs/freeze_signal_loss.json` | `summary.by_group.blade_only.comb_line_delta_db_median` | 7.798 |
| [^92] | `outputs/freeze_signal_loss.json` | `summary.by_group.why_ko` | 동체가 든 열(A·B·2 초 호버·세 엔진)은 동체 DC 가 크고 가짜 변조도 거기에 실려 있었다… |
| [^93] | `outputs/freeze_signal_loss.json` | `verdict.open_ko` | ⚠ `report07_hover_long*` 두 열에는 **순수 PO 대조 팔이 없다**. 2 초… |
| [^94] | `outputs/meshfix_attack.json` | `recommended_gate_before_any_sigma_claim` | (6행 표) |


---

## 절 2. 스톡 솔버와 맞대면 «면이 많아서 에코가 커진다» 가설은 반증되고, 런타임의 96.9% 는 호스트가 쓴다



> ### 한 일
> **같은 메쉬를 스톡 경로 솔버에 그대로 넣고 경로 수·진폭·런타임을 우리 커널과 같은 카드에서 나란히 쟀다.**

### 결과
1. 삼각형 29,932 개 [^95] (mavic4pro)에서 광선 예산 2.6e+08 spp [^96] 일 때 스톡 경로는 자세당 213.8 개 [^97] 다.
2. 이미지법 정반사 경로는 36 자세 [^98] 를 통틀어 2 개 [^99] (자세 1 개 [^100])다 — 나머지 진폭은 확산 항이 낸다.
3. «면이 많아서 에코가 커진다» 가설은 스톡 기울기 0.093 dB/decade [^101] 로 REFUTED [^102] 다.
4. 평판 대조군에서 정반사 진폭은 무한거울 값 -75.37 dB [^103] 에 고정되어, 판 크기를 40 배 [^104] 로 키워도 7.4e-07 dB [^105] 안에 머문다.
5. 런타임은 같은 통계로 나란히 적는다 — S(스톡 PathSolver) 72.8 [^106]~128.0 ms [^107] (중앙값 106.3 ms [^108] · 12 설정 [^109]) 대 B(우리 per-pose) 9.2 [^110]~356.2 ms [^111] (중앙값 38.1 ms [^112]) 다. 이 대조가 통제하는 것은 하드웨어이고, 두 팔이 재는 양은 경로 해와 σ 로 갈린다 [^113].


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 같은 메쉬 | 우리 커널이 먹는 그 메쉬를 스톡 경로 솔버에 그대로 넣는다 — 면 수·광선예산·자세 격자를 맞춘다 |
| 면 수 축 | 같은 형상을 테셀레이션만 바꿔 쌓고 스톡 에코의 기울기를 dB/decade 로 잰다 |
| 평판 대조군 | 닫힌형 무한거울 진폭이 있는 평판에서 판 크기만 키운다 — 경로 진폭이 면적을 보는지가 여기서 갈린다 |
| 런타임 | 같은 카드에서 스톡 PathSolver 와 우리 per-pose 를 나란히 재고, 비용을 호스트/GPU 단계로 쪼갠다 |

### 재현

```bash
PYTHONPATH=src python benchmark/facet_count.py
PYTHONPATH=src python benchmark/runtime_benchmark.py
PYTHONPATH=src python src/build_part04_kernel.py
```

| | |
|---|---|
| 출력 | `outputs/facet_count.json`, `outputs/facet_mechanism.json`, `outputs/runtime_benchmark.json` |
| 소요 | 약 40분 (GPU 1장 — 스톡 솔버와 우리 커널을 같은 카드에서 돌린다) |
| 비고 | 런타임은 같은 카드·같은 세션에서 재야 비교가 선다 |

---


## 같은 메쉬를 스톡에 그대로 넣으면

삼각형 29,932 개 [^95] (mavic4pro)에서 광선 예산 2.6e+08 spp [^96] 일 때 경로는 자세당 213.8 개 [^97] 다. 그중 **이미지법 정반사** 경로는 36 자세 [^98] 를 통틀어 2 개 [^99] (자세 1 개 [^100])뿐이다.

나머지 진폭은 확산 항이 낸다 — 그 항은 정규화 산란패턴을 쓰는 경험 모델이라 표면전류 적분이 아니다. 그래서 σ 를 얹을 자리가 생긴다.


## «면이 많아서 에코가 커진다» 를 반증했다

평판 대조군에서 정반사 진폭은 무한거울 값 -75.37 dB [^103] 에 고정되어, 판 크기를 40 배 [^104] 로 키워도 7.4e-07 dB [^105] 안에 머문다. 같은 축을 드론 메쉬에서 재면 스톡 기울기가 0.093 dB/decade [^101] 이고, 판정은 **REFUTED [^102]** 다.

⭐ S(스톡) 쪽에서 면 수가 값을 옮기는 자리는 **이미지법 중복 경로** 하나다 — 한 변 1 m [^114] 금속 평판을 2 [^115] → 512 삼각형 [^116] 으로 쪼개면 코히런트 전력이 9.54 dB [^117] 오르고, 늘어난 경로는 진폭·위상·지연이 같은 복사본이라 합이 20·log10(N) 을 0.0017 dB [^118] 안에서 따른다 [^119].

⭐ 이 반증이 리포트 2 의 결론과 같은 물건이다 — 경로 진폭은 면적·곡률·치수를 인자로 받지 않는다. 그 인자 목록은 [리포트 2 절 4 «필드 갱신 인자 여덟 개에 면적·곡률·치수·λ…»](02_stock-engine.ipynb) 에 있다.


## 비용은 어디에 있나

같은 카드에서 같은 통계로 나란히 쟀다 — S(스톡 PathSolver) 72.8 [^106]~128.0 ms [^107] (12 설정 [^109]) 대 B(우리 per-pose) 9.2 [^110]~356.2 ms [^111], 중앙값끼리는 106.3 [^108] 대 38.1 ms [^112] 다.

⚠ 두 팔은 다른 양을 잰다 — S 는 씬의 경로 해, B 는 표적 σ 이고 씬 내용도 갈린다. 이 대조가 통제하는 것은 하드웨어다 [^113]. 사다리는 자릿수를 놓는 것이고 같은 양을 잰 속도비는 그 뒤의 일이다 [^120].

그 비용의 96.9% [^121] 가 호스트에 있고 GPU 광선추적은 3.1% [^122] 다. PO 단계가 48.5% [^123] 이므로 디바이스로 옮길 자리는 거기다 — 전격자 재생성 추정은 0.91 h [^124] 다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| PO 면적분을 디바이스 커널로 옮긴다 | 전격자 재생성 비용이 확정된다 — 지금 호스트 몫이 대부분이다 | `src/rcs_sbr.py` → `outputs/runtime_benchmark.json` 재측정 |
| 면 수 축을 표적 사다리의 운동학 축과 분리해 다시 읽는다 | 형상 정밀도가 σ 에 주는 몫이 광선예산 몫과 갈린다 | [리포트 7 절 3 «모양의 유무는 수십 dB 를 가르고»](07_size-law.ipynb) |
| 기준해와 맞대 커널의 구현오차를 확정한다 | 이 커널이 PO 를 제대로 계산하는지가 kr 전 구간에서 확정된다 | **절 4** «해석 PO 구 대비 구현오차는 kr 전 구간에…» |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 30개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^95] | `outputs/facet_count.json` | `levels[0].n_tri` | 29932 |
| [^96] | `outputs/facet_count.json` | `meta.spp_main` | 256000000 |
| [^97] | `outputs/facet_count.json` | `levels[0].rt_n_paths_mean` | 213.8 |
| [^98] | `outputs/facet_count.json` | `levels[0].spec_n_aspects` | 36 |
| [^99] | `outputs/facet_count.json` | `levels[0].spec_n_paths_total` | 2 |
| [^100] | `outputs/facet_count.json` | `levels[0].spec_n_aspects_nonzero` | 1 |
| [^101] | `outputs/facet_count.json` | `slopes.stock_incoh` | 0.093 |
| [^102] | `outputs/facet_count.json` | `verdict.result` | REFUTED |
| [^103] | `outputs/facet_mechanism.json` | `theory.amp_image_source_db` | -75.37 |
| [^104] | `outputs/facet_mechanism.json` | `VERDICT.plate_size_invariance.size_ratio_max` | 40 |
| [^105] | `outputs/facet_mechanism.json` | `VERDICT.plate_size_invariance.rt_spread_db` | 7.419e-07 |
| [^106] | `outputs/runtime_benchmark.json` | `answer.same_card_control.stock_sionna_pathsolver_ms.min` | 72.8 |
| [^107] | `outputs/runtime_benchmark.json` | `answer.same_card_control.stock_sionna_pathsolver_ms.max` | 128 |
| [^108] | `outputs/runtime_benchmark.json` | `answer.same_card_control.stock_sionna_pathsolver_ms.median` | 106.3 |
| [^109] | `outputs/runtime_benchmark.json` | `answer.same_card_control.stock_sionna_pathsolver_ms.n_configs` | 12 |
| [^110] | `outputs/runtime_benchmark.json` | `production_per_pose.summary_ms.min` | 9.24 |
| [^111] | `outputs/runtime_benchmark.json` | `production_per_pose.summary_ms.max` | 356.2 |
| [^112] | `outputs/runtime_benchmark.json` | `production_per_pose.summary_ms.median` | 38.13 |
| [^113] | `outputs/runtime_benchmark.json` | `answer.same_card_control.caveat` | different quantities (paths vs RCS) and different scene… |
| [^114] | `outputs/report00_evidence.json` | `H_tessellation_changes_the_answer.numbers.per_side[1].side_m` | 1 |
| [^115] | `outputs/report00_evidence.json` | `H_tessellation_changes_the_answer.numbers.per_side[1].n_tri[0]` | 2 |
| [^116] | `outputs/report00_evidence.json` | `H_tessellation_changes_the_answer.numbers.per_side[1].n_tri[4]` | 512 |
| [^117] | `outputs/report00_evidence.json` | `H_tessellation_changes_the_answer.numbers.max_inflation_db` | 9.542 |
| [^118] | `outputs/report00_evidence.json` | `H_tessellation_changes_the_answer.numbers.coherent_N_law_max_resid_db` | 0.001717 |
| [^119] | `outputs/report00_evidence.json` | `H_tessellation_changes_the_answer.claim` | 같은 1 m 금속 평판을 2 → 512 삼각형으로 쪼개기만 해도 코히어런트 전력이 9.54 dB 부… |
| [^120] | `outputs/runtime_benchmark.json` | `answer.what_the_measurement_does_NOT_support[2]` | the stock-Sionna baselines measure a different quantity… |
| [^121] | `outputs/runtime_benchmark.json` | `answer.cost_structure.host_side_pct` | 96.86 |
| [^122] | `outputs/runtime_benchmark.json` | `production_per_pose.stage_pct_median_over_configs.rt_trace` | 3.141 |
| [^123] | `outputs/runtime_benchmark.json` | `production_per_pose.stage_pct_median_over_configs.po` | 48.5 |
| [^124] | `outputs/runtime_benchmark.json` | `production_per_pose.whole_published_grid.projected_hours` | 0.9128 |


---

## 절 3. 수신 방향 그림자 광선을 켜면 상반성 위반이 9.69 → 8.24 dB 로 내려간다



> ### 한 일
> **각 충돌점에서 수신기 방향으로 그림자 광선을 한 번 더 쏘아 출사 쪽 가림을 판정하고, 그 효과를 상반성으로 쟀다.**

### 결과
1. 상반성 위반 최대치(전 β 최악)가 9.69 [^125] → 8.24 dB [^126] 로 내려간다.
2. 모노스태틱에서 이 검사는 무연산이라 생산 σ 는 0.000e+00 dB [^127] 그대로다 — 켜도 옛 결과가 안 움직인다.
3. 바이스태틱 자세 패턴은 β ≤ 45° [^128] 에서 성립한다 — 그 범위의 상반성 RMS 가 2.57 dB [^129] 다.
4. 이 단계는 Sagitta(preprint, arXiv:2604.09243 각주 1)가 바이스태틱 SBR 에서 빠져 있다고 지목한 바로 그 단계다 — 선행이 이름 붙인 구멍을 메운 자리다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 출사 가시성 | 각 충돌점에서 수신기 방향으로 그림자 광선을 한 번 더 쏜다 (`src/rcs_sbr.py` `rcs_sbr_multistatic()`) |
| 무엇으로 재나 | **상반성** — 보내는 자리와 받는 자리를 맞바꿔도 σ 가 같아야 한다는 성질이다. 위반량이 곧 모형오차의 크기다 |
| 모노스태틱 대조 | 송수신이 같은 자리면 이 검사가 무연산이 되는지 확인한다 — 옛 생산 σ 가 안 움직여야 한다 |
| 유효창 | β 를 키우며 상반성 RMS 를 재서 자세 패턴을 주장할 범위를 못박는다 |

### 재현

```bash
PYTHONPATH=src python benchmark/verify_sbr_defect_fixes.py
PYTHONPATH=src python src/build_part04_kernel.py
```

| | |
|---|---|
| 출력 | `outputs/sbr_defect_fixes.json` |
| 소요 | 약 25분 (GPU 1장) |
| 비고 | 상반성은 정리(theorem)라 위반량이 그대로 모형오차의 하한이다 |

---


## 무엇을 한 번 더 쏘나

각 충돌점에서 수신기 방향으로 그림자 광선을 한 번 더 쏘아 **출사 쪽 가림**을 판정한다(`src/rcs_sbr.py` `rcs_sbr_multistatic()`). 조명 쪽만 보면 «빛이 닿는 면» 까지는 맞지만, 그 면이 수신기에서 보이는지는 따로 물어야 한다.

Sagitta(preprint, arXiv:2604.09243 각주 1)가 바이스태틱 SBR 에서 빠져 있다고 지목한 바로 그 단계다.


## 켠 효과 — 상반성으로 잰다

**상반성**은 보내는 자리와 받는 자리를 맞바꿔도 σ 가 같아야 한다는 성질이다. 정리이므로 위반량이 그대로 모형오차의 크기다. 위반 최대치가 9.69 [^125] → 8.24 dB [^126] 로 내려간다.

모노스태틱에서 이 검사는 무연산이라 생산 σ 는 0.000e+00 dB [^127] 그대로다 — 새 단계를 켜도 옛 모노스태틱 결과가 한 눈금도 안 움직인다는 뜻이다.


## 두 계열을 나란히 — β 별 RMS

헤드라인 9.69 [^125] → 8.24 dB [^126] 는 **전 β 최악값** 하나의 개선이다. β 별 RMS 로 내려가면 두 방향이 함께 있다.

| β [°] | 출사 가시성 끔 [dB] | 출사 가시성 켬 [dB] |
|---|---|---|
| 15 [^130] | 1.36 [^131] | 0.79 [^132] |
| 30 [^133] | 2.72 [^134] | 1.57 [^135] |
| 45 [^128] | 2.36 [^136] | 2.57 [^129] |
| 60 [^137] | 2.22 [^138] | 1.67 [^139] |
| 75 [^140] | 1.83 [^141] | 2.84 [^142] |
| 90 [^143] | 6.11 [^144] | 4.02 [^145] |

β 45° [^128] (주장 창의 경계)와 75° [^140] 두 칸에서는 켠 계열이 더 크다 — 창 규칙의 문턱 RMS 는 그 경계 칸의 값으로 세워져 있다. 나머지 네 칸과 전 β 최악값에서는 켠 쪽이 더 작다.


## 자세 패턴을 주장할 범위

**바이스태틱 자세 패턴은 β ≤ 45° [^128] 에서 성립한다** — 창 규칙은 «경계 안 상반성 RMS ≤ 2.57 dB [^129]» 이고, 경계는 선언값이다. RMS 는 β 에 단조가 아니라 β 60° 에서 1.67 [^139] 로 내려왔다가 β 90° 에서 4.02 dB [^145] 까지 오른다. 그 위의 β 는 창 밖이고, 검출 기하가 이 창 안에 드는지는 [리포트 12 절 2 «TX·RX·표적 배치와 β·앙각·원거리장이 유…»](12_observability.ipynb) 가 확인한다.

전방산란 쪽 끝(β→180°)은 조명 게이트와 수신 게이트가 상호배타라 σ ≡ 0 이 된다 — 주장 창을 후방~중간 바이스태틱각으로 못박는 이유이고, 그 목록은 **절 6** «커널이 아직 못 하는 것은 편파 분리·PTD·…» 에 있다.


## 이 절의 수치가 서 있는 자리

⚠ 위 상반성 표는 각도 모양 |Γ(θ)| 가 바이스태틱 경로에 들어오기 **전** 커널의 산출이다. 지금 그 경로는 각도 모양을 곱하고, 같은 씬에서 복소장과 σ 두 경로가 1e-14 dB [^146] 안에서 맞는다 [^147] — 상반성 표는 그 커널로 다시 낼 대상이다.

⚠ 그리고 **생산 검출 사슬은 이 커널을 부르지 않는다** — σ 를 모노스태틱 격자에서 이등분선 방향으로 조회한다. 그 근사의 크기가 β 15° 에서 rms 중앙 6.39 [^148] · p95 최대 18.9 dB [^149] 이므로, 검출 결과가 인용하는 β 가 몇 도인지가 그 근사의 유효성을 정한다 ([리포트 12 절 2 «TX·RX·표적 배치와 β·앙각·원거리장이 유…»](12_observability.ipynb)).


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 2회 반사를 β 별로 다시 돌린다 | 바이스태틱 유효범위가 45° 위로 얼마나 넓어지는지 확정된다 | `src/rcs_sbr.py` `rcs_sbr_multistatic()` 상반성 검사 |
| 검출 기하가 이 창 안에 드는지 확인한다 | 결과 편이 인용하는 β 가 자세 패턴 유효창 안인지가 확정된다 | [리포트 12 절 2 «TX·RX·표적 배치와 β·앙각·원거리장이 유…»](12_observability.ipynb) |
| 대칭 사입사 인자 √((n̂·û_i)(n̂·û_s)) 를 grazing 안정화와 함께 다시 시도한다 | 상반성 위반이 격자 폭발 없이 줄어드는지가 확정된다 — 지금은 표준 (n̂·û_i) 를 두고 σ 레벨 기하평균(`symmetrize=True`)으로 우회한다 | `src/rcs_sbr.py` `rcs_sbr_multistatic()` obliquity 주석 [^150] |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 26개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^125] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.worst_without_exit_vis_db` | 9.687 |
| [^126] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.worst_with_exit_vis_db` | 8.237 |
| [^127] | `outputs/sbr_defect_fixes.json` | `d4_epsilon_sensitivity.combos[0].by_drone.mavic4pro.monostatic_noop_max_abs_db` | 0 |
| [^128] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.beta_deg[3]` | 45 |
| [^129] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.rms_with_exit_vis_db[3]` | 2.568 |
| [^130] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.beta_deg[1]` | 15 |
| [^131] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.rms_without_exit_vis_db[1]` | 1.357 |
| [^132] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.rms_with_exit_vis_db[1]` | 0.7897 |
| [^133] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.beta_deg[2]` | 30 |
| [^134] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.rms_without_exit_vis_db[2]` | 2.718 |
| [^135] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.rms_with_exit_vis_db[2]` | 1.569 |
| [^136] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.rms_without_exit_vis_db[3]` | 2.357 |
| [^137] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.beta_deg[4]` | 60 |
| [^138] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.rms_without_exit_vis_db[4]` | 2.225 |
| [^139] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.rms_with_exit_vis_db[4]` | 1.671 |
| [^140] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.beta_deg[5]` | 75 |
| [^141] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.rms_without_exit_vis_db[5]` | 1.832 |
| [^142] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.rms_with_exit_vis_db[5]` | 2.836 |
| [^143] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.beta_deg[6]` | 90 |
| [^144] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.rms_without_exit_vis_db[6]` | 6.106 |
| [^145] | `outputs/sbr_defect_fixes.json` | `d2_exit_vis_effect_on_reciprocity.rms_with_exit_vis_db[6]` | 4.024 |
| [^146] | `outputs/verify_bistatic_field.json` | `verdict.sigma_angle_gamma_gap_db` | 1.421e-14 |
| [^147] | `outputs/verify_bistatic_field.json` | `kernel.angle_gamma_wired` | 예 |
| [^148] | `outputs/geometry_grid.json` | `sigma_transfer.bisector_approximation_error_by_beta.15.dsigma_rms_median_db` | 6.394 |
| [^149] | `outputs/geometry_grid.json` | `sigma_transfer.bisector_approximation_error_by_beta.15.dsigma_p95_max_db` | 18.87 |
| [^150] | `outputs/audit_rcs_kernel.json` | `q2_mono_bi_multistatic.obliquity_question.code_says` | rcs_sbr.py:663-666 · 581-584 — 대칭 obliquity √((n̂·û_i)(… |


---

## 절 4. 해석 PO 구 대비 구현오차는 kr 전 구간에서 λ/16 격자 0.201 dB · 생산 λ/12 격자 0.254 dB 안이다



> ### 한 일
> **구 후방산란의 닫힌형 기준해 둘과 이면각 닫힌형에 커널을 맞대 구현오차와 모형 간극을 따로 쟀다.**

### 결과
1. 해석 PO 구 대비 최대 편차가 kr 1 [^151]~100 [^152] 전 구간에서 λ/16 격자 0.201 [^153] · 생산 λ/12 격자 0.254 dB [^154] 다(입사 48 방향 [^155]).
2. 정확 Mie 대비 최대 편차는 λ/16 격자 6.73 [^156] · 생산 λ/12 격자 6.60 dB [^157] (kr=1) 이고, 이쪽이 PO 라는 모형 자체의 간극이다 — 격자를 209 배 [^158] 조여도 -0.054 dB [^159] 움직인다.
3. kr ≥ 30 산포는 λ/16 격자에서 해석 PO 대비 0.885% [^160] · Mie 대비 1.834% [^161] 이고, 생산 λ/12 격자에서 1.939% [^162] · 1.547% [^163] 다.
4. PEC 이면각 닫힌형 8πa²b²/λ² 와는 2회 반사에서 최대 0.556 dB [^164] 다 — 다중반사 위상이 맞는다는 뜻이다.
5. 기체 7 × 밴드 3 = 21 조합 [^165] 중 1 개 [^166] 가 Mie 기준 1 dB 문턱 아래에 놓이고, 그것이 실측 대상 DJI Mini 5 Pro [^167] 다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 기준해 둘 | 구에 대해서만 맥스웰 방정식이 그대로 풀리는 **정확 Mie** 와, 같은 구에 PO 근사를 적용해 손으로 푼 **해석 PO** 다 (`benchmark/mie_pec_sphere.py:98`, `:127`) |
| 왜 둘인가 | (커널 − Mie) = (커널 − 해석 PO) + (해석 PO − Mie) 다. 앞항은 **구현오차**(격자를 조이면 준다), 뒷항은 **모형오차**(격자로는 안 준다) |
| 다중반사 | 직각 이면각 이등분선 입사의 닫힌형 8πa²b²/λ² 와 변 길이 4점에서 맞댄다(`benchmark/verify_sbr_defect_fixes.py`, λ/12 격자) |
| 자기검사 | 상반성 σ(û_i,û_s)=σ(û_s,û_i) 위반을 기체에서 잰다 — 정리 위반이 곧 모형오차다 |

### 재현

```bash
PYTHONPATH=src python benchmark/sbr_kr_sweep.py
PYTHONPATH=src python benchmark/verify_sbr_defect_fixes.py
PYTHONPATH=src python src/build_part04_kernel.py
```

| | |
|---|---|
| 출력 | `outputs/sbr_kr_sweep.json`, `outputs/sbr_defect_fixes.json`, `outputs/report00_po_case.json`, `outputs/report02_derived.json` |
| 소요 | 약 1시간 (GPU 1장 — kr 스윕이 대부분이다) |
| 비고 | 두 기준해는 우리 출력이 아니라 과녁이다 |

---


## 과녁이 둘이고, 재는 것이 다르다

구 후방산란은 두 개의 **닫힌형 기준해**(근사 없이 식으로 바로 값이 나오는 답)를 갖는다 — 구에 대해서만 맥스웰 방정식이 그대로 풀리는 **정확 Mie** 와, 같은 구에 PO 근사를 적용해 손으로 푼 **해석 PO** 다. 둘 다 우리 출력이 아니라 과녁이다.

```
(커널 − Mie)  =  (커널 − 해석 PO)   +   (해석 PO − Mie)
                  ↑ 우리 수치오차          ↑ PO 모델 자체의 간극
```
커널이 PO 이므로 **수치 수렴의 과녁은 해석 PO** 이고, Mie 잔차는 PO 모델 자체의 간극이라는 두 번째 눈금이다. 둘을 나눠 두면 각각이 얼마인지 그대로 읽힌다.


## 일곱 기체가 놓인 자리에서 각각 얼마인가

![report02_f5_reference_gap](../outputs/figures/report02_f5_reference_gap.png)

**그림 2.** 일곱 기체가 놓인 kr 자리에서 우리 수치오차와 PO 모델의 간극은 각각 얼마인가?


## 두 눈금 — 격자 두 개에서

|  | 우리 수치오차 · 기준해 = 해석 PO | PO 모델의 간극 · 기준해 = 정확 Mie |
|---|---|---|
| 최대 편차 (kr=1..100) · λ/16 | 0.201 dB [^153] | 6.73 dB [^156] (kr=1) |
| 최대 편차 (kr=1..100) · 생산 λ/12 | 0.254 dB [^154] | 6.60 dB [^157] (kr=1) |
| kr≥30 산포 · λ/16 | 0.885% [^160] | 1.834% [^161] |
| kr≥30 산포 · 생산 λ/12 | 1.939% [^162] | 1.547% [^163] |
| 1 dB 안으로 드는 kr | 전 구간 (kr=1 [^151] 부터) | kr ≥ 9.06 [^168] |
| 0.5 / 0.2 dB 안으로 | 전 구간 | 15.16 [^169] / 30.87 [^170] |

격자를 두 줄로 적는 이유는 사슬이 둘이기 때문이다 — 커널 기본은 λ/12(`src/rcs_sbr.py` `DEFAULT_DIV`)이고, σ 앵커 사슬은 λ/16 으로 돈다(`benchmark/rcs_anchor.py` `raw_sigma_az(div=16)`).


## 검증 3층 — 무엇을 각각 재는가

| 층 | 과녁 | 무엇을 재나 | 결과 |
|---|---|---|---|
| ① | 해석 PO 구 | 커널 구현 | 최대 λ/16 0.201 [^171] · 생산 λ/12 0.254 dB [^154] |
| ② | PEC 구 Mie 정확해 | PO 라는 모형 | ka=1 에서 -6.58 dB [^172] · 광학영역 산포 λ/16 1.83 [^173] · 생산 λ/12 1.55% [^163] |
| ③ | 얇은 띠 2D EFIE MoM | 가는 특징 | 가장 가는 폭에서 TM -4.02 [^174] · TE +7.53 dB [^175] |
| + | PEC 이면각 닫힌형 | 다중반사 위상 | 2-bounce 최대 0.556 dB [^176] |
| + | 상반성 정리 | 정리 위반 = 모형오차 | 기체 최악 8.24 dB [^177] (같은 검사를 인쇄한 선행 0편) |


## 이면각 — 오목부에서 오는 항

**이면각**은 두 평판이 90° 로 맞붙은 표준 형상이고, **PEC** 는 전기를 완벽히 통하는 이상적 금속이다. 직각 이면각의 이등분선 입사는 σ = 8πa²b²/λ² 로 닫혀 있다. 2회 반사를 켜고 변 길이 4점에서 그 값과 맞댔다(3.5 GHz, λ/12 격자).

| 변 a [m] | 해석해 [dBsm] | 1회 반사 [dBsm] | 2회 반사 [dBsm] | 오차 [dB] |
|---|---|---|---|---|
| 0.15 | 2.39 | -14.63 | 2.95 | +0.556 |
| 0.20 | 7.39 | -13.79 | 7.85 | +0.466 |
| 0.30 | 14.43 | -118.33 | 14.51 | +0.076 |
| 0.40 | 19.43 | -7.76 | 19.32 | -0.108 |

출처 [^178]

1회/2회 열이 오목부에서 오는 항이 어디에 있는지 그대로 보여준다. 셋째 줄(a = 0.30 m)의 1회 반사는 -118.3 dBsm [^179] 로 무너진 널이고, 그 줄의 작은 오차는 그 우연한 상쇄 위에 앉아 있다 — 이 표에서 인용할 값은 4점의 **최대** 0.556 dB [^164] 다.

같은 스크립트가 매끄러운 기준체도 함께 잰다 — 구는 λ/16 격자에서 해석 PO 대비 -0.021 dB [^180], 평판은 λ/10 격자에서 -0.011 dB [^181] 다.


## 이 눈금이 드론에 그대로 걸리는가

기체 7 × 밴드 3 = 21 조합 [^165] 중 1 개 [^166] 가 Mie 기준 1 dB 문턱 아래에 놓이고, 그것이 실측 대상 DJI Mini 5 Pro [^167] 다.

⚠ **이 kr 눈금은 매끄러운 구에서만 맞는 눈금이다** — 구는 몸 전체가 하나의 넓은 곡면이지만 드론은 얇은 판과 가는 막대의 모음이라, PO 가 어긋나는 자리를 정하는 것은 기체 전체 크기가 아니라 **부품 하나의 폭이 파장에 비해 얼마나 넓은가** 다. 그 세 번째 눈금이 **절 5** «PO 유효 무릎을 부품 폭으로 옮기면 어느 부품이 어느 밴드에서 떨어지는지가 보인다» 다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 평판·이면각 표준체로 같은 kr 스윕을 돌린다 | 얇고 모서리 많은 표적에서의 PO 간극 문턱이 선다 | `benchmark/verify_sbr_defect_fixes.py` 의 두 닫힌형 재사용 |
| 부품 폭 눈금으로 옮겨 우리 세 밴드가 어디에 서는지 읽는다 | 어느 부품이 어느 밴드에서 무릎 아래인지가 확정된다 | **절 5** «PO 유효 무릎을 부품 폭으로 옮기면 어느 부…» |
| 상용 솔버 한 대와 같은 형상에서 교차검증한다 | 구·이면각 밖의 형상에서 구현오차가 확정된다 | `OPENSOURCE.md` — RadarSimPy 교차검증 항목 |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 31개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^151] | `outputs/sbr_kr_sweep.json` | `summary_div16.kr_min` | 1 |
| [^152] | `outputs/report00_po_case.json` | `s3_validation.layer1_analytic_po_convergence.kr_sweep_kr_max` | 100 |
| [^153] | `outputs/sbr_kr_sweep.json` | `summary_div16.max_abs_db_vs_po` | 0.2006 |
| [^154] | `outputs/sbr_kr_sweep.json` | `summary_div12.max_abs_db_vs_po` | 0.2544 |
| [^155] | `outputs/report00_po_case.json` | `s3_validation.layer1_analytic_po_convergence.kr_sweep_n_incidence` | 48 |
| [^156] | `outputs/sbr_kr_sweep.json` | `summary_div16.max_abs_db_vs_mie` | 6.729 |
| [^157] | `outputs/sbr_kr_sweep.json` | `summary_div12.max_abs_db_vs_mie` | 6.599 |
| [^158] | `outputs/report00_po_case.json` | `s3_validation.layer1_analytic_po_convergence.sphere_ka1_grid_refine_factor` | 209.4 |
| [^159] | `outputs/report00_po_case.json` | `s3_validation.layer2_pec_sphere_mie.improvement_from_refining_grid_db` | -0.05373 |
| [^160] | `outputs/sbr_kr_sweep.json` | `summary_div16.std_sbr_over_po_pct_kr_ge30` | 0.8855 |
| [^161] | `outputs/sbr_kr_sweep.json` | `summary_div16.std_sbr_over_mie_pct_kr_ge30` | 1.834 |
| [^162] | `outputs/sbr_kr_sweep.json` | `summary_div12.std_sbr_over_po_pct_kr_ge30` | 1.939 |
| [^163] | `outputs/sbr_kr_sweep.json` | `summary_div12.std_sbr_over_mie_pct_kr_ge30` | 1.547 |
| [^164] | `outputs/sbr_defect_fixes.json` | `d3_multibounce_phase.max_abs_err_db` | 0.5563 |
| [^165] | `outputs/report02_derived.json` | `electrical.n_airframe_band` | 21 |
| [^166] | `outputs/report02_derived.json` | `electrical.n_below_po_1db` | 1 |
| [^167] | `outputs/report02_derived.json` | `electrical.kr_min_name` | DJI Mini 5 Pro |
| [^168] | `outputs/report02_derived.json` | `po_floor.kr_below_1p0_db` | 9.06 |
| [^169] | `outputs/report02_derived.json` | `po_floor.kr_below_0p5_db` | 15.16 |
| [^170] | `outputs/report02_derived.json` | `po_floor.kr_below_0p2_db` | 30.87 |
| [^171] | `outputs/report00_po_case.json` | `s3_validation.layer1_analytic_po_convergence.kr_sweep_max_abs_db_vs_po_div16` | 0.2006 |
| [^172] | `outputs/report00_po_case.json` | `s3_validation.layer2_pec_sphere_mie.po_minus_mie_at_ka1_db` | -6.584 |
| [^173] | `outputs/report00_po_case.json` | `s3_validation.layer2_pec_sphere_mie.kr_sweep_std_pct_vs_mie_kr_ge30_div16` | 1.834 |
| [^174] | `outputs/report00_po_case.json` | `s3_validation.layer3_thin_plate_2d_mom.po_minus_tm_at_0p15lam_db` | -4.022 |
| [^175] | `outputs/report00_po_case.json` | `s3_validation.layer3_thin_plate_2d_mom.po_minus_te_at_0p15lam_db` | 7.535 |
| [^176] | `outputs/report00_po_case.json` | `s3_validation.layer4_dihedral_multibounce.max_abs_err_2bounce_db` | 0.5563 |
| [^177] | `outputs/report00_po_case.json` | `s3_validation.layer5_reciprocity_selfcheck.drone_worst_violation_db` | 8.237 |
| [^178] | `outputs/sbr_defect_fixes.json` | `d3_multibounce_phase.rows` | (4행 표) |
| [^179] | `outputs/report00_po_case.json` | `s3_validation.layer4_dihedral_multibounce.a03_sbr_1bounce_dbsm` | -118.3 |
| [^180] | `outputs/sbr_defect_fixes.json` | `d3_multibounce_phase.sphere_and_plate.sphere_vs_po_db.sphere_lam/16_vs_po` | -0.02084 |
| [^181] | `outputs/sbr_defect_fixes.json` | `d3_multibounce_phase.sphere_and_plate.plate_db.plate_lam/10` | -0.01114 |


---

## 절 5. PO 유효 무릎을 부품 폭으로 옮기면 어느 부품이 어느 밴드에서 떨어지는지가 보인다



> ### 한 일
> **얇은 금속 판을 2D 적률법 참값과 PO 로 각각 내어 유효 무릎을 폭으로 정하고, 그 무릎을 한 기체의 부품 치수로 옮겨 주파수 축에 세웠다.**

### 결과
1. 무릎의 정의는 «두 편파 중 나쁜 쪽의 |PO−MoM| 이 1 dB 를 넘는 구간의 상단» 이고, 그 아래로 내려가려면 특징 폭이 0.729 λ [^182] 이상이어야 한다.
2. 그 무릎을 부품 치수로 옮기면 동체 2.68 GHz [^183] · 암뿌리 4.86 GHz [^184] · 암끝 7.28 GHz [^185] · 프로펠러 15.86 GHz [^186] · 모터 15.97 GHz [^187] · 캐노피 35.13 GHz [^188] · PCB 73.08 GHz [^189] 에서 통과한다.
3. 우리 생산 3 밴드는 전부 이 문턱 아래에 부품을 남긴다 — 무릎 위로 올라선 특징이 LTE 1.843 GHz [^190] 에서는 0 개(동체 무릎이 2.68 GHz [^183])이고, 5G 3.5 GHz [^191] 에서는 동체 하나, WiFi 5.21 GHz [^192] 에서는 동체와 암뿌리(4.86 GHz [^184]) 둘이다.
4. 절대 σ 에는 격자 불확도도 붙는다 — λ/16 서브셀 디더 산포 1.78 dB [^193] 다.
5. 저대역 판정 라벨 `B_PO_LIMIT [^194]` 에 대한 적대검증 판정은 `PREMATURE [^195]` 다 — PO 한계는 **부호만** 확인됐고 크기 귀속은 열려 있다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 참값 | 얇은 금속 판을 2D 적률법(MoM — 맥스웰 방정식을 수치로 푸는 방법)으로 낸다. 두 편파를 따로 낸다 |
| 편파란 | 전파의 전기장이 흔들리는 방향이다 — 판의 긴 축과 나란한 쪽을 TM, 그에 수직인 쪽을 TE 라 부른다 |
| 무릎의 정의 | 두 편파 중 **나쁜 쪽**의 \|PO−MoM\| 이 1 dB 를 넘는 구간의 상단. 나쁜 쪽을 쓰는 것이 보수적이다 |
| 주파수로 옮기기 | Phantom 3 급 한 기체의 부품 치수를 넣어 부품마다 몇 GHz 에서 무릎을 넘는지 계산한다 |

### 재현

```bash
PYTHONPATH=src python benchmark/lowfreq_anchor.py
PYTHONPATH=src python src/build_part04_kernel.py
```

| | |
|---|---|
| 출력 | `outputs/lowfreq_anchor.json`, `outputs/lowfreq_attack.json`, `outputs/report00_po_case.json`, `outputs/report02_derived.json` |
| 소요 | 약 15분 (GPU 0장 — 2D MoM 은 CPU 다) |
| 비고 | 치수는 Phantom 3 를 사진 실측으로 다시 짓기 전 스윕에서 인용한 값이다 |

---


## 무릎을 무엇으로 정했나

얇은 금속 판을 참값(2D 적률법 MoM)과 PO 로 각각 내고 맞대면, 두 편파 중 나쁜 쪽의 차이가 1 dB 아래로 내려가는 문턱이 **폭 ≥ 0.729 λ [^182]** 다. 정의는 «max(|PO−MoM TM|,|PO−MoM TE|) 가 1 dB 를 넘는 구간의 상단» [^196] 이다.

⭐ 이 눈금은 **절 4** «해석 PO 구 대비 구현오차는 kr 전 구간에…» 의 두 눈금과 **다른 것을 잰다.** (커널 − 해석 PO) 는 우리 구현이 PO 를 제대로 계산하는지의 눈금이고, 폭 0.15 λ [^197] 인 얇은 판에서도 격자를 조이면 0.015 dB [^198] 까지 수렴한다. (PO − 참값) 은 PO 라는 모델 자체가 참값과 떨어진 거리이고, 이 편이 크기를 준 것이 그쪽이다.


## 부품마다 몇 GHz 에서 무릎을 넘나

| 부품 | 폭 | 무릎 주파수 |
|---|---|---|
| 동체 | 81.51 mm | 2.68 GHz [^183] |
| 암뿌리 | 45 mm | 4.86 GHz [^184] |
| 암끝 | 30 mm | 7.28 GHz [^185] |
| 프로펠러 | 13.78 mm | 15.86 GHz [^186] |
| 모터 | 13.68 mm | 15.97 GHz [^187] |
| 캐노피 | 6.22 mm | 35.13 GHz [^188] |
| PCB | 2.99 mm | 73.08 GHz [^189] |

⚠ 이 목록은 **기체 하나**의 치수이고 7기체 공통이 아니다 — 크기 폭 4.66 배 [^199] 안에서 S1000+ 처럼 큰 기체는 이 문턱이 그만큼 낮은 주파수로, Mini 급은 그만큼 높은 주파수로 옮겨간다.


## 우리 세 밴드는 어디에 서 있나

⚠⚠ **우리 생산 3 밴드(LTE 1.843 [^190] · 5G 3.5 [^191] · WiFi 5.21 GHz [^192])는 전부 이 문턱 아래에 부품을 남긴다** — LTE 에서는 동체까지 아래이고, 5G 에서는 동체 하나가 무릎 위, WiFi 에서는 동체와 암뿌리(4.86 GHz [^184]) 둘이 무릎 위다. 가장 높은 밴드에서도 암끝·프로펠러·모터·캐노피·PCB 는 아래에 있다.

문헌 측정의 위쪽 끝(18.2 GHz [^200])까지 **줄곧** 문턱 아래에 남는 부품은 캐노피와 PCB 둘뿐이다. 프로펠러와 모터는 그 끝에 닿기 전인 15.86 GHz [^186] · 15.97 GHz [^187] 에서 문턱을 넘는다 — 문헌 대역의 맨 위 토막에서만 넘는 셈이다.

그래서 이 저장소는 σ 의 절대 크기 대신 **각도 구조와 밴드 간 상대 순위**를 주장한다. 절대 레벨은 [리포트 15 절 2 «구가 σ 를 절대량으로 만들고»](15_measurement.ipynb) 의 교정구가 측정으로 앵커한다.


## 부호는 한 방향을 가리킨다

지배채널(TM) 기준 PO 는 얇은 특징을 **과소평가**한다 — 가장 가는 시험 폭에서 TM 기준 -4.02 dB [^201] (음수 = 우리가 낮다), TE 기준 +7.53 dB [^202] 다. 참값 자체가 편파로 11.56 dB [^203] 갈린다.

따라서 우리 저주파 σ 는 낮게 나와 있을 개연성이 크고, 그 방향이라면 검출 성능 산출물은 **보수적(비관적)** 쪽으로 틀렸다. 스칼라 PO 는 편파를 **못** 가르므로 TE 채널 기준으로는 부호가 반대다 — 방향을 못 박으려면 편파 있는 커널이 필요하다 [^204].


## 이 결과로 말할 수 없는 것 셋

· ✗ 「저주파 σ 가 틀렸다」 → 맞는 말은 **불확도가 지금 선언된 것보다 크고 그 크기를 아직 정하는 중이다** [^205].

· ✗ 「고대역은 검증됐다」 → 캐노피·PCB 는 문헌 측정 대역의 위쪽 끝까지 문턱 아래에 머물고, 프로펠러·모터도 그 끝 바로 아래에서야 문턱을 넘는다 [^206].

· ✗ 「격자를 더 촘촘히 하면 σ 가 고쳐진다」 → **정반대다.** 격자를 조이면 저대역 기울기가 오히려 더 가팔라진다 — 이것이 이 라운드에서 가장 확실한 결과다 [^207].

절대 σ 에는 격자 불확도도 함께 붙는다 — λ/16 서브셀 디더 산포 1.78 dB [^193] 다. 저대역 판정 라벨 `B_PO_LIMIT [^194]` 의 적대검증 판정은 `PREMATURE [^195]` 이고, 살아남은 것은 표본화 배제뿐이다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 편파를 가르는 커널로 넓히고 VV/HH 를 따로 낸다 | 부호가 못 박히고 저대역 불확도의 크기가 확정된다 | `src/materials.py:171` 편파 분해 → [리포트 14 절 5 «교정된 절대 σ 를 만드는 조건은 여섯 항목이…»](14_robustness.ipynb) |
| 생산 σ 를 `ptd=True` 로 다시 낸다 | 모서리 항이 밴드 기울기를 얼마나 옮기는지가 수치로 남는다 | `benchmark/rcs_anchor.py --ptd` · 비용 47.2% [^208] |
| 7기체 각각의 부품 치수로 무릎 표를 다시 낸다 | 기체마다 어느 밴드에서 어느 부품이 떨어지는지가 전수로 확정된다 | `benchmark/lowfreq_anchor.py` → 기체 루프 추가 |
| 교정구를 표적과 같은 자리에서 함께 잰다 | 지금 우리 PO 출력인 절대 레벨이 처음으로 측정에 앵커된다 | [리포트 15 절 2 «구가 σ 를 절대량으로 만들고»](15_measurement.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 27개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^182] | `outputs/lowfreq_anchor.json` | `thin_plate.truth_2d_mom_fine_width_grid.knee_a_over_lam` | 0.7289 |
| [^183] | `outputs/lowfreq_attack.json` | `q5_blast_radius.po_validity_blast_radius_the_real_one.recomputed_by_me_frequency_at_which_each_feature_passes_that_knee.body_81.51mm` | 2.68 |
| [^184] | `outputs/lowfreq_attack.json` | `q5_blast_radius.po_validity_blast_radius_the_real_one.recomputed_by_me_frequency_at_which_each_feature_passes_that_knee.arm_root_45mm` | 4.86 |
| [^185] | `outputs/lowfreq_attack.json` | `q5_blast_radius.po_validity_blast_radius_the_real_one.recomputed_by_me_frequency_at_which_each_feature_passes_that_knee.arm_tip_30mm` | 7.28 |
| [^186] | `outputs/lowfreq_attack.json` | `q5_blast_radius.po_validity_blast_radius_the_real_one.recomputed_by_me_frequency_at_which_each_feature_passes_that_knee.prop_blade_13.78mm` | 15.86 |
| [^187] | `outputs/lowfreq_attack.json` | `q5_blast_radius.po_validity_blast_radius_the_real_one.recomputed_by_me_frequency_at_which_each_feature_passes_that_knee.motor_13.68mm` | 15.97 |
| [^188] | `outputs/lowfreq_attack.json` | `q5_blast_radius.po_validity_blast_radius_the_real_one.recomputed_by_me_frequency_at_which_each_feature_passes_that_knee.canopy_6.22mm` | 35.13 |
| [^189] | `outputs/lowfreq_attack.json` | `q5_blast_radius.po_validity_blast_radius_the_real_one.recomputed_by_me_frequency_at_which_each_feature_passes_that_knee.pcb_2.99mm` | 73.08 |
| [^190] | `outputs/report02_derived.json` | `bands_ghz.LTE` | 1.843 |
| [^191] | `outputs/report02_derived.json` | `bands_ghz.5G` | 3.5 |
| [^192] | `outputs/report02_derived.json` | `bands_ghz.WiFi` | 5.21 |
| [^193] | `outputs/report00_po_case.json` | `s2_our_kernel.grid_dither.dither_spread_div16_db` | 1.782 |
| [^194] | `outputs/report00_po_case.json` | `s4_limits.adversarial_verdict_verbatim.attacked_verdict_label` | B_PO_LIMIT |
| [^195] | `outputs/report00_po_case.json` | `s4_limits.adversarial_verdict_verbatim.adversarial_verdict` | PREMATURE |
| [^196] | `outputs/report00_po_case.json` | `s4_limits.po_validity_knee_rule` | max(\|PO−MoM TM\|,\|PO−MoM TE\|) 가 1 dB 를 넘는 구간의 상단 |
| [^197] | `outputs/lowfreq_anchor.json` | `thin_plate.per_width.0.15.a_lam` | 0.15 |
| [^198] | `outputs/lowfreq_anchor.json` | `thin_plate.per_width.0.15.max_abs_vs_po_two_finest_db` | 0.01493 |
| [^199] | `outputs/report02_derived.json` | `mesh.span_ratio` | 4.662 |
| [^200] | `outputs/p3_validation_v2.json` | `slope.das_published.band[1]` | 18.2 |
| [^201] | `outputs/report00_po_case.json` | `s3_validation.layer3_thin_plate_2d_mom.po_minus_tm_at_0p15lam_db` | -4.022 |
| [^202] | `outputs/report00_po_case.json` | `s3_validation.layer3_thin_plate_2d_mom.po_minus_te_at_0p15lam_db` | 7.535 |
| [^203] | `outputs/report00_po_case.json` | `s3_validation.layer3_thin_plate_2d_mom.tm_minus_te_at_0p15lam_db` | 11.56 |
| [^204] | `outputs/report00_po_case.json` | `s4_limits.our_production_bands_vs_knee.sign_of_the_error` | 지배채널(TM) 기준 PO 는 얇은 특징을 **과소평가**한다(prop 0.083λ 에서 −7.25… |
| [^205] | `outputs/lowfreq_attack.json` | `q5_blast_radius.what_must_not_be_said[0]` | '저주파 σ 가 틀렸다' — 아니다. '저주파 σ 의 불확도가 선언된 것보다 훨씬 크고, 그 크기를… |
| [^206] | `outputs/lowfreq_attack.json` | `q5_blast_radius.what_must_not_be_said[1]` | '고대역(6~18.2 GHz)은 검증됐다' — 아니다. a_high 의 주파수-잔차 SE 가 0.0… |
| [^207] | `outputs/lowfreq_attack.json` | `q5_blast_radius.sampling_blast_radius_actual.direction` | 생산 σ 는 1.843 GHz 근처에서 약 0.3 dB **높게** 나와 있고, 저대역 기울기는 약… |
| [^208] | `outputs/ptd_wiring.json` | `verdict.cost_increase_pct` | 47.17 |


---

## 절 6. 커널이 아직 못 하는 것은 편파 분리·PTD·재테셀레이션·다중반사 Γ(θ) 넷이고, 각각의 크기를 적었다



> ### 한 일
> **커널의 열린 항목을 하나씩 세고 각 항목이 σ 를 얼마나 움직이는지를 dB 로 함께 적었다.**

### 결과
1. 편파 — 면적분이 스칼라다. |Γ(θ)| 는 TE·TM 전력평균이라 채널 분리가 따로 없고, 가장 가는 시험 폭에서 참값이 TM−TE 11.56 dB [^209] 로 갈린다.
2. 모서리 프린지(PTD) — 배선은 있고 생산 경로는 `ptd=False` 다. 켜면 TE 가 +7.53 [^210] → +10.81 dB [^211] 로 벌어지고, 비용은 +47.2% [^212] 다.
3. 테셀레이션 — 입력 메쉬를 어떻게 쪼갰는가에 σ 가 따라간다. 같은 드론 메쉬를 삼각형 29,932 [^213]→450 개 [^214] 로 쌓아 재면 삼각형 수 10 배당 B(SBR+PO) 1.65 [^215] · P(순수 PO) 1.92 dB/decade [^216] 다. 메쉬 사다리는 앵커 물체 두 점에서 돌렸다.
4. 2회 이상 다중반사 — 생산 σ 는 1-bounce 다. 변 0.15 m [^217] 이면각에서 1-bounce -14.63 [^218] ↔ 2-bounce 2.95 dBsm [^219] 다.
5. Γ(θ) 각도 모양 — 1회 반사 네 경로(σ 격자·복소장·바이스태틱 둘)가 같은 함수를 기본으로 곱하고 순수 PO 는 호출 옵트인이며 [^220], 다중반사 경로는 아직 수직입사 값 그대로다. 켠 경로에서 프롭 채널이 +5.16 [^221] ~ +6.48 dB [^222] 움직였다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 무엇을 세나 | 커널이 계산에 넣지 않는 항과, 넣되 생산 경로에서 꺼 둔 항을 따로 센다 — 둘은 성질이 다르다 |
| 크기를 어떻게 붙이나 | 항마다 그 항이 σ 를 움직이는 양을 이미 잰 대조에서 끌어온다. 크기를 못 붙인 항은 그렇게 적는다 |
| 왜 표인가 | 열린 항목을 산문으로 쓰면 사과가 되고 표로 쓰면 예산이 된다 — 다음 라운드가 집어 갈 수 있는 형태다 |

### 재현

```bash
PYTHONPATH=src python src/build_part04_kernel.py
```

| | |
|---|---|
| 출력 | `outputs/report00_po_case.json`, `outputs/ptd_wiring.json`, `outputs/report00_evidence.json` |
| 소요 | 약 1분 (GPU 0장 — 이미 잰 값을 모은 표다) |
| 비고 | 각 행의 크기는 그 항을 실제로 켜거나 끄고 잰 대조에서 왔다 |

---


## 열린 항목과 그 크기

커널이 계산에 넣지 않는 항과, 넣되 생산 경로에서 꺼 둔 항을 한 표에 센다 — 항마다 크기는 그 항을 실제로 켜거나 끄고 잰 대조에서 왔다.


| 열린 항목 | 현재 상태 | 크기 |
|---|---|---|
| 편파 | 면적분이 스칼라다 — \|Γ(θ)\| 는 TE·TM 전력평균이라 채널 분리가 따로 없다 | 가장 가는 시험 폭에서 참값이 TM−TE 11.56 dB [^209] 로 갈린다 |
| Γ(θ) 각도 모양의 근사 | 벌크 각도 모양을 얇은 판 근사로 곱한다 — 수직입사에서 보정값과 동일하다 [^223] | 프롭 채널 레벨 +5.16 [^221] ~ +6.48 dB [^222] · 기체 3대의 전체 드론 σ 는 방위평균 +0.036 [^224] ~ +0.328 dB [^225] · 칸별 최대 1.27 [^226] ~ 6.00 dB [^227] |
| Γ(θ) 다중반사 경로 | 1회 반사 네 경로는 기본 켬이고 순수 PO 는 호출 옵트인이다. 남은 자리는 `rcs_sbr(max_bounce≥2)` 로, 재질 키를 받고도 각도 모양을 곱하지 않는다 [^228] | 그 경로로 낸 2회 반사 σ 는 왼쪽 위 행의 크기만큼 1회 반사 경로와 갈린다 |
| 모서리 프린지(PTD) | 배선은 있고 생산 경로는 `ptd=False` 다 | 켜면 TE 가 +7.53 [^210] → +10.81 dB [^211] 로 벌어진다 |
| 2회 이상 다중반사 | 생산 σ 는 1-bounce 다 — 오목부의 항이 통째로 빠진다 | 변 0.15 m [^217] 이면각에서 1-bounce -14.63 [^218] ↔ 2-bounce 2.95 dBsm [^219] (**절 4** 표의 a = 0.30 m 줄은 1회 반사가 무너진 널이라 크기 인용에 쓰지 않는다) |
| 크리핑파·표면파 | 우리 커널과 Sionna 가 같은 자리에 선다 — GO/UTD 계열 고주파 근사의 바깥이다 | 매끄러운 볼록체 그림자 경계를 감아 도는 성분 |
| 전방산란 β→180° | 조명 게이트와 수신 게이트가 상호배타라 σ ≡ 0 이 된다 | 주장 창을 후방~중간 바이스태틱각으로 못박는다 |
| 생산 경로의 참값 대조 | 참값 앵커는 penetrate=False · \|Γ\|=1 · 볼록이다 | 생산은 penetrate=True · 재질 Γ · 자기가림 · 1-bounce 다 |
| 테셀레이션 축 | 커널의 스위치가 아니라 **입력 메쉬**의 축이다 — 메쉬 사다리는 앵커 물체 두 점에서 돌렸다 | 삼각형 수 10 배당 B(SBR+PO) 1.65 [^215] · P(순수 PO) 1.92 dB/decade [^216] · 형상이 보존된 수준만 쓰면 0.27 [^229] · 0.65 dB/decade [^230] |
| 회전 프로펠러 도플러 | Sionna 의 Paths.doppler 는 객체당 강체 속도 1벡터다 | 부품별 위상은 우리 커널의 복소 E 에서 온다 |


## 이 표를 읽는 법

편파 · Γ(θ) 두 항 · PTD · 2회 이상 다중반사 다섯 줄은 **커널 자신의 축**이다. **편파**는 커널이 애초에 안 가르는 축이라 크기가 참값 쪽에서 오고, **Γ(θ)** 는 1회 반사 경로와 다중반사 경로가 갈라져 있다 — 그 항의 근거는 교과서 프레넬과 θ=0 비트동일 둘뿐이고, 각도 경로를 밟는 과녁은 아직 세우지 않았다(다음 단계 표). **PTD** 는 배선이 끝나 있고 스위치만 꺼 둔 항이라 켜면 바로 값이 나온다 — 비용 +47.2% [^212] 가 그 값이다. **테셀레이션**은 커널 밖 **입력 메쉬**의 축이라 크기를 우리 두 팔(B·P)의 기울기로 적었고, 그 축을 정면으로 다룬 것이 [리포트 7 «표적 사다리»](07_size-law.ipynb) 다.

전방산란 · 크리핑파 · 참값 대조 · 회전 프로펠러 도플러 넷은 **주장 창을 좁히는 방식**으로 이미 처리돼 있다 — 전방산란은 창 밖으로 내보냈고(그 창은 **절 3** «수신 방향 그림자 광선을 켜면 상반성 위반이…» 가 β ≤ 45° 로 못 박았다), 크리핑파는 고주파 근사 계열 전체가 못 내는 항이라 Sionna 도 같은 자리에 선다. 회전 프로펠러 도플러는 [리포트 8 «마이크로도플러»](08_1_scene.ipynb) 가 슬로타임 재추적으로 우회한다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 편파 분해를 커널에 넣고 VV/HH 를 따로 낸다 | 표의 첫 행이 크기에서 값으로 바뀐다 | `src/materials.py:171` → [리포트 14 절 5 «교정된 절대 σ 를 만드는 조건은 여섯 항목이…»](14_robustness.ipynb) |
| Γ(θ) 를 다중반사 경로에 배선하고 오목부 기체에서 다시 낸다 | 2회 반사 σ 가 1회 반사 경로와 같은 각도 물리 위에 선다 | `src/rcs_sbr.py` `rcs_sbr(max_bounce≥2)` |
| 각도 경로를 실제로 밟는 과녁을 하나 세운다 — 유전체 평판 각도 스윕 대 MoM·프레넬 | Γ(θ) 의 외부 눈금이 처음 생긴다 — 지금 과녁은 전부 PEC 라 각도 경로를 안 밟는다 | [^231] |
| 생산 σ 를 `ptd=True` 로 다시 낸다 | 모서리 항이 밴드 기울기를 얼마나 옮기는지가 수치로 남는다 | `benchmark/rcs_anchor.py --ptd` → `outputs/rcs_anchor_ptd.json` |
| 테셀레이션 축을 앵커 물체 밖 형상으로 넓힌다 | 메쉬 쪼개기가 σ 를 얼마나 부풀리는지가 드론 형상에서 확정된다 | [리포트 7 절 3 «모양의 유무는 수십 dB 를 가르고»](07_size-law.ipynb) |
| 2회 반사를 생산 경로에서 켜고 비용과 이득을 함께 잰다 | 오목부 있는 기체에서 1-bounce 가정의 대가가 확정된다 | `src/rcs_sbr.py` `rcs_sbr_multistatic()` |
| 회전 블레이드 마이크로도플러를 이 커널 위에서 검증한다 | 미세도플러 서명을 이 커널의 산출물로 인용할 수 있게 된다 | [리포트 8 절 2 «시간표본마다 자세를 새로 놓고 다시 쏘아 슬로…»](08_3_pattern.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 23개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^209] | `outputs/report00_po_case.json` | `s3_validation.layer3_thin_plate_2d_mom.tm_minus_te_at_0p15lam_db` | 11.56 |
| [^210] | `outputs/report00_po_case.json` | `s3_validation.layer3_thin_plate_2d_mom.po_minus_te_at_0p15lam_db` | 7.535 |
| [^211] | `outputs/report00_po_case.json` | `s3_validation.layer3_thin_plate_2d_mom.po_ptd_minus_te_at_0p15lam_db` | 10.81 |
| [^212] | `outputs/ptd_wiring.json` | `verdict.cost_increase_pct` | 47.17 |
| [^213] | `outputs/facet_count.json` | `levels[0].n_tri` | 29932 |
| [^214] | `outputs/facet_count.json` | `levels[5].n_tri` | 450 |
| [^215] | `outputs/facet_count.json` | `slopes.sbr` | 1.654 |
| [^216] | `outputs/facet_count.json` | `slopes.po` | 1.92 |
| [^217] | `outputs/sbr_defect_fixes.json` | `d3_multibounce_phase.rows[0].a_m` | 0.15 |
| [^218] | `outputs/sbr_defect_fixes.json` | `d3_multibounce_phase.rows[0].sbr_1bounce_dbsm` | -14.63 |
| [^219] | `outputs/sbr_defect_fixes.json` | `d3_multibounce_phase.rows[0].sbr_2bounce_dbsm` | 2.947 |
| [^220] | `outputs/angle_gamma_po_impact.json` | `_meta.switch` | rcs_po: 호출 옵트인(mats=점별 재질 키) ∧ 킬스위치 ANGLE_GAMMA(env SIO… |
| [^221] | `outputs/angle_gamma_impact.json` | `propeller_channel_el_-15_3p5GHz.matrice4e.level_delta_db` | 5.16 |
| [^222] | `outputs/angle_gamma_impact.json` | `propeller_channel_el_-15_3p5GHz.mini5pro.level_delta_db` | 6.48 |
| [^223] | `outputs/angle_gamma_impact.json` | `_meta.design` | \|Γ(θ)\| = \|Γ_보정\| · \|Γ_벌크(θ)\|/\|Γ_벌크(0)\| — 수직입사에서 비트 동일 |
| [^224] | `outputs/angle_gamma_sigma_impact.json` | `mini5pro/el+15.delta_db` | 0.03607 |
| [^225] | `outputs/angle_gamma_sigma_impact.json` | `phantom4/el-15.delta_db` | 0.3283 |
| [^226] | `outputs/angle_gamma_sigma_impact.json` | `mini5pro/el+15.cell_max_abs_db` | 1.266 |
| [^227] | `outputs/angle_gamma_sigma_impact.json` | `phantom4/el+15.cell_max_abs_db` | 6.001 |
| [^228] | `outputs/audit_rcs_kernel.json` | `q3_angle_gamma_wiring.UNWIRED_paths.rcs_sbr (max_bounce 경로)` | src/rcs_sbr.py:813 에서 matk 를 받아놓고 안 쓴다 |
| [^229] | `outputs/facet_count.json` | `slopes.sbr_shapeok` | 0.2744 |
| [^230] | `outputs/facet_count.json` | `slopes.po_shapeok` | 0.6537 |
| [^231] | `outputs/audit_rcs_kernel.json` | `open_items_ranked[2].item` | Γ(θ) 의 **외부 앵커** — 지금 0 이다. 기준해가 전부 PEC 라 각도경로를 한 번도 안… |
